# Anomalie w sieci kanalizacyjnej

Wyniki `python main.py anomalie` (folder `wyniki/anomalie/`).

**Jak działa:**
1. Dla każdego obiektu i doby liczymy cechy: nadwyżka przepływu ponad **bazę bezdeszczową**, udział przepływu nocnego (infiltracja),
   **residual energii** (rzeczywiste − prognoza modelu, który danego okresu nie widział), kWh/m³ (sprawność pomp), opad, temperatura, roztopy.
2. Trzy niezależne modele – **Isolation Forest, LOF, One-Class SVM** – oceniają, czy doba jest nietypowa.
   **Anomalia = zgadzają się co najmniej 2 z 3.** Opad jest w cechach celowo: dużo wody przy ulewie to norma, dużo wody bez deszczu – anomalia.
3. Reguły nadają typ: dopływ opadowy, roztopy, infiltracja / dopływ bez deszczu, spadek (awaria / przerwa), spadek sprawności pomp.
4. **Tryb panelowy** (jak w Detektorze Anomalii GUS): jednostka = pompownia, okres = doba. `jump_score` – nagła zmiana względem
   własnych 14 dób, `zscore_miedzy` – odstawanie od pozostałych pompowni tego samego dnia (ta sama pogoda). Z nich `Rodzaj_Anomalii`.
5. „Oczekiwany odsetek anomalii” dobierany jest na **alarmach tłoczni w Chorzowie** – jedynych potwierdzonych zdarzeniach w danych.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

A = Path("wyniki") / "anomalie"
dni = pd.read_csv(A / "dni.csv", parse_dates=["dzien"])
anom = pd.read_csv(A / "anomalie.csv", parse_dates=["dzien"])
rank = pd.read_csv(A / "ranking.csv")
print(f"dób: {len(dni):,} | anomalii: {len(anom)} ({100 * len(anom) / dni['glosy'].notna().sum():.1f}% ocenionych dób)")

## 1. Walidacja na alarmach tłoczni CH
`zlapane_*` – ile dni z alarmem model wskazał (±1 doba). `trafienia_przy_alarmie` – ile wskazań wypadło przy alarmie. `lift` – ile razy lepiej niż losowy wybór dni.

In [ ]:
p = A / "walidacja_CH.csv"
if p.exists():
    w = pd.read_csv(p)
    w["odsetek"] = w["odsetek"].map("{:.0%}".format)
    display(w.round(1))
else:
    print("brak alarmów do walidacji")

## 2. Czy modele się zgadzają
Ile dób wskazał każdy model i ile razy wskazania się pokryły. Zespół bierze tylko doby wskazane przez ≥ 2 modele.

In [ ]:
ocen = dni[dni["glosy"].notna()]
print(ocen[["IF", "LOF", "SVM"]].sum().rename("wskazanych dób").to_string())
print("\nliczba głosów na dobę:")
print(ocen["glosy"].value_counts().sort_index().rename("dób").to_string())
fig, ax = plt.subplots(figsize=(8, 3.5))
ocen.groupby("obiekt_id")["glosy"].value_counts().unstack(fill_value=0).drop(columns=0, errors="ignore").plot.bar(stacked=True, ax=ax)
ax.set_title("Doby z 1, 2 i 3 głosami per obiekt (anomalia = 2 lub 3)"); ax.set_xlabel(""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. Typy anomalii per obiekt

In [ ]:
t = anom.groupby(["obiekt_id", "typ"]).size().unstack(fill_value=0)
display(t)
t.plot.barh(stacked=True, figsize=(10, 4)); plt.title("Anomalie wg typu"); plt.xlabel("dób"); plt.tight_layout(); plt.show()

## 4. Ranking obiektów – wody obce
`udzial_wod_obcych_proc` – jaka część przepływu to woda ponad bazę bezdeszczową. `reakcja_m3_na_mm` – ile m³ wód obcych na 1 mm deszczu. `noc_udzial_sucho` – przepływ nocny / średni w suche dni (wysoki = infiltracja).

In [ ]:
display(rank.round(2))
display(Image(filename=str(A / "ranking.png")))

## 4a. Rodzaj anomalii – tryb panelowy
**Nagła zmiana** – doba odbiega od poprzednich 14 dób tej pompowni. **Odstaje od innych** – tego samego dnia (przy tej samej pogodzie) pompownia odbiega od pozostałych bardziej niż zwykle. To drugie to mocny sygnał problemu lokalnego: nieszczelności, nielegalnego podłączenia albo awarii.

In [ ]:
r = anom.groupby(["obiekt_id", "Rodzaj_Anomalii"]).size().unstack(fill_value=0)
display(r)
display(pd.crosstab(anom["typ"], anom["Rodzaj_Anomalii"]))

## 5. Co wyróżnia anomalie
Dla każdej anomalii liczymy odporny z-score każdej cechy (o ile odstaje od mediany obiektu, w jednostkach IQR). **Główna przyczyna** = cecha „skutku” (woda, energia, pompy) z największym |z|. Pogoda jest kontekstem, a nie przyczyną. Prawy wykres: na ile cecha odróżnia anomalie od zwykłych dób.

In [ ]:
p = A / "przyczyny.csv"
if p.exists():
    display(pd.read_csv(p, index_col=0).round(1))
    display(Image(filename=str(A / "przyczyny.png")))
    display(pd.read_csv(A / "przyczyny_per_typ.csv", index_col=0))

## 6. Anomalie z opisem
Jedno zdanie na anomalię, liczone w Pythonie (typ, główna przyczyna, skala, opad, liczba głosów).

In [ ]:
OBIEKT = None      # <- ZMIEŃ np. "RU_W"; None = wszystkie
a = anom if OBIEKT is None else anom[anom["obiekt_id"] == OBIEKT]
a = a.assign(s=a["skala_m3"].abs().fillna(a["skala_kwh"].abs())).sort_values("s", ascending=False)
for o in a["opis"].head(15):
    print(o)

## 7. Anomalie na osi czasu
Czarna linia – rzeczywiste, zielona – baza bezdeszczowa, kropki – anomalie wg typu, niebieskie słupki – opad. Dla CH czerwone pasy to alarmy tłoczni.

In [ ]:
for p in sorted(A.glob("anomalie_*.png")):
    display(Image(filename=str(p)))

## 8. Największe anomalie
Posortowane wg skali: m³ wód ponad normę (gdzie jest przepływ) albo kWh ponad prognozę.

In [ ]:
kol = ["obiekt_id", "dzien", "typ", "Rodzaj_Anomalii", "glowna_cecha", "z_glownej", "glosy", "skala_m3", "skala_kwh", "nadwyzka_m3_proc", "residual_proc", "opad_3d"]
top = anom.assign(skala=anom["skala_m3"].abs().fillna(anom["skala_kwh"].abs())).sort_values("skala", ascending=False)
display(top[kol].head(20))

## 10. Raport dla dyspozytora
Po `python main.py raport`. Liczby policzone w Pythonie, model językowy (Ollama) tylko formułuje zdania – jak w Detektorze Anomalii GUS. Bez Ollamy raport zawiera same fakty.

In [ ]:
from IPython.display import Markdown
p = A / "raport.md"
display(Markdown(p.read_text(encoding="utf-8")) if p.exists() else "brak – uruchom: python main.py raport")

## 9. Znane zdarzenia
Czy detektor zauważył zdarzenia znane z eksploracji (powódź IX 2024, przeciążenie RU_W w styczniu 2024).

In [ ]:
p = A / "znane_zdarzenia.csv"
display(pd.read_csv(p) if p.exists() and p.stat().st_size > 1 else "brak")